# Pairs trading & cointegration

**pyportfolios.com research** · EWA / EWC, Jan 2010 – Dec 2024 · NumPy · Pandas · statsmodels · Matplotlib · yfinance

The static Engle–Granger methodology, end to end, on the classic pair
(iShares Australia vs Canada — two commodity-driven markets):

1. OLS hedge ratio on log price levels,
2. ADF unit-root tests on each leg and on the residual,
3. OU half-life of the spread,
4. the band backtest — enter |z| > 2, exit |z| < 0.5, 5 bp per leg change,
5. the honest rerun with a rolling out-of-sample hedge ratio.

(The *dynamic* hedge-ratio treatment of this same pair — Kalman filtering —
is its own tutorial and notebook.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
from statsmodels.regression.rolling import RollingOLS
from statsmodels.tsa.stattools import adfuller, coint

plt.rcParams["figure.figsize"] = (10, 4.5)

## 1 · Data: two markets tethered by commodities

In [ ]:
px = yf.download(["EWA", "EWC"], start="2010-01-01", end="2025-01-01",
                 auto_adjust=True, progress=False)["Close"].dropna()
ly, lx = np.log(px["EWC"]), np.log(px["EWA"])
px.plot(title="EWA vs EWC — adjusted closes");

## 2 · Engle–Granger step 1: the hedge ratio

Regress one log price on the other. The slope β is the number of units of EWA
that immunise one unit of EWC against the common trend.

In [ ]:
ols = sm.OLS(ly, sm.add_constant(lx)).fit()
alpha, beta = ols.params.iloc[0], ols.params.iloc[1]
spread = ly - beta * lx - alpha
print(f"hedge ratio beta = {beta:.3f}   intercept = {alpha:.3f}")

## 3 · Step 2: unit-root tests

Each log price should be non-stationary (ADF cannot reject a unit root), while
the residual should be stationary (ADF rejects). `coint` runs the same recipe
with the correct Engle–Granger critical values, which are stricter than plain
ADF because β was itself estimated.

In [ ]:
for name, s in [("log EWA", lx), ("log EWC", ly), ("EG residual", spread)]:
    stat, pval, *_ = adfuller(s, regression="c", autolag="AIC")
    print(f"ADF {name:<12} stat {stat:7.3f}   p-value {pval:.4f}")

eg_stat, eg_p, _ = coint(ly, lx)
print(f"\nEngle-Granger coint test: stat {eg_stat:.3f}   p-value {eg_p:.4f}")

## 4 · How fast does it snap back? The OU half-life

Fit dS = θ·S·dt + noise on the residual: the AR(1) coefficient gives the speed
of mean reversion, and half-life = −ln 2 / θ. This number sets the natural
holding period — and the z-score window should comfortably exceed it.

In [ ]:
ds, lag = spread.diff().dropna(), spread.shift(1).dropna()
ou = sm.OLS(ds, sm.add_constant(lag.loc[ds.index])).fit()
theta = ou.params.iloc[1]
print(f"half-life = {-np.log(2)/theta:.1f} trading days")

## 5 · The band backtest — and the honest rerun

Enter when |z| > 2, flatten when |z| < 0.5, pay 5 bp on every change of
position. Then repeat with a hedge ratio estimated on a rolling 252-day window
lagged one day — no future prices inside β.

In [ ]:
Z_WIN, ENTRY, EXIT, COST = 60, 2.0, 0.5, 5e-4

def band_backtest(spread):
    z = (spread - spread.rolling(Z_WIN).mean()) / spread.rolling(Z_WIN).std(ddof=1)
    state = np.where(z < -ENTRY, 1.0, np.where(z > ENTRY, -1.0, np.nan))
    pos = pd.Series(state, index=z.index)
    pos[z.abs() < EXIT] = 0.0
    pos = pos.ffill().fillna(0.0)
    pnl = pos.shift(1) * spread.diff()
    net = pnl - pos.diff().abs().shift(1) * COST
    sh = lambda x: np.sqrt(252) * x.dropna().mean() / x.dropna().std(ddof=1)
    entries = int(((pos != 0) & (pos.shift(1) == 0)).sum())
    return z, pos, net, sh(pnl), sh(net), entries

z, pos, net, g_sh, n_sh, entries = band_backtest(spread)
yrs = len(spread) / 252
print(f"gross Sharpe {g_sh:.2f}   net Sharpe {n_sh:.2f}   trades/yr {entries/yrs:.1f}")

# the honest version: beta from a rolling window, lagged one day
roll = RollingOLS(ly, sm.add_constant(lx), window=252).fit()
b = roll.params.shift(1)
spread_oos = (ly - b.iloc[:, 1] * lx - b.iloc[:, 0]).dropna()
*_, net_oos_sh, entries_oos = band_backtest(spread_oos)[2:]
print(f"rolling-beta OOS net Sharpe {net_oos_sh:.2f}   "
      f"trades/yr {entries_oos/(len(spread_oos)/252):.1f}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
z.plot(ax=ax[0], color="#4a4a42", lw=0.8)
for lvl in (2, -2): ax[0].axhline(lvl, color="#0a8a8a", ls="--", lw=1)
ax[0].set_title("spread z-score with ±2σ entry bands")
net.fillna(0).cumsum().plot(ax=ax[1], color="#0a8a8a")
ax[1].set_title("cumulative net P&L (spread units, 5 bp per leg change)")
plt.tight_layout();

## Takeaways

- Both legs are I(1); the Engle–Granger residual is stationary at conventional
  levels — the pair is cointegrated over this sample.
- The half-life tells you the natural holding period before you place a trade.
- The in-sample full-period β flatters the backtest; the rolling out-of-sample
  rerun is the number you should believe — and it is materially lower.
- Multiply that haircut by the number of pairs you scanned before settling on
  this one: that is what the Deflated Sharpe Ratio formalises.

*© pyportfolios.com — runnable companion to the article. Data: Yahoo Finance via yfinance.*